# RLHF

## 1. ¿Por qué un modelo de lenguaje necesita alinearse?

Un modelo de lenguaje como GPT-2 aprende a predecir cuál es la siguiente palabra más probable en un texto. Eso lo hace bueno completando oraciones, pero no necesariamente bueno respondiendo de forma útil o apropiada.

Por ejemplo, si le preguntas *"¿cómo puedo mejorar mi estado de ánimo?"*, el modelo puede responder con algo que aparece frecuentemente en internet aunque sea una mala recomendación, simplemente porque estadísticamente es probable y no porque sea correcto.

La **alineación** es el proceso de corregir eso: guiar al modelo para que sus respuestas sean más útiles, seguras y acordes con lo que los humanos realmente prefieren.

---

## 2. RLHF y métodos relacionados

**RLHF** significa **Reinforcement Learning from Human Feedback**. Es un enfoque para alinear modelos de lenguaje usando preferencias humanas sobre distintas respuestas. La idea es sencilla: si para una misma pregunta una respuesta es mejor que otra, esa preferencia se usa como señal para enseñar al modelo a responder mejor en el futuro.

El proceso clásico de RLHF suele dividirse en tres etapas. Primero, se realiza un **ajuste supervisado (SFT)** para darle al modelo una base inicial de respuestas útiles. Después, se entrena un **Reward Model**, que aprende a puntuar respuestas a partir de comparaciones humanas entre una opción preferida y otra rechazada. Finalmente, el modelo principal se optimiza para generar respuestas que obtengan mejor puntuación según ese evaluador, sin alejarse demasiado de su comportamiento original.

Con el tiempo aparecieron variantes que simplifican este esquema. **RLAIF** reemplaza parte de la retroalimentación humana por evaluaciones hechas por otros modelos ya alineados. Por su parte, métodos como **DPO, IPO y ORPO** evitan el paso explícito de aprendizaje por refuerzo y reformulan el problema como entrenamiento directo sobre pares de respuestas preferida y rechazada.

En este notebook se utiliza **DPO (Direct Preference Optimization)**. Este método ajusta el modelo para dar más probabilidad a la respuesta preferida y menos a la rechazada, sin necesidad de ejecutar un ciclo completo de aprendizaje por refuerzo. Además, incorpora un control para evitar que el modelo cambie demasiado respecto al original. Ese control está regulado por el parámetro **β**, que define qué tan libre o qué tan conservador será el ajuste.

---

## 3. Pipeline de este notebook

```text
┌─────────────────────────────────────────────────────────────────┐
│                   PIPELINE DE ESTE NOTEBOOK                     │
│                                                                 │
│  PARTE 0 — Entorno y datos                                      │
│  Instalar dependencias · cargar hh-rlhf · preparar formato DPO │
│                         ↓                                       │
│  PARTE 1 — Modelo base                                          │
│  Cargar GPT-2 · ver cómo responde antes del entrenamiento       │
│                         ↓                                       │
│  PARTE 2 — Entrenamiento DPO                                    │
│  Configurar DPOTrainer · entrenar · monitorear la loss         │
│                         ↓                                       │
│  PARTE 3 — Evaluación                                           │
│  Comparar base vs DPO · Reward Model como juez externo         │
│                         ↓                                       │
│  PARTE 4 — Actividad práctica                                   │
│  Experimentar con hiperparámetros · cambiar modelo · analizar  │
└─────────────────────────────────────────────────────────────────┘

---
##  Parte 0 — Entorno y datos


In [ ]:
import subprocess, sys

# Versiones compatibles con Kaggle GPU (Python 3.12, CUDA 12.x, 2025)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'trl>=0.8.6,<0.9',
    'transformers>=4.43.0,<4.46',
    'accelerate>=0.33.0',
    'datasets>=2.20.0',
])

# Verificar
import importlib
import trl, transformers, datasets as _ds
importlib.reload(trl); importlib.reload(transformers)


In [ ]:
import warnings, textwrap, os
import numpy as np
import matplotlib.pyplot as plt
import torch

import trl

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset, Dataset

# trl >= 0.8.0 usa DPOConfig; < 0.8.0 usa TrainingArguments + beta explícito
try:
    from trl import DPOTrainer, DPOConfig
    _HAS_DPO_CONFIG = True
    print('Modo: DPOConfig ')
except ImportError:
    from trl import DPOTrainer
    _HAS_DPO_CONFIG = False

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

DEVICE_ID = 0 if torch.cuda.is_available() else -1
TORCH_DEV = torch.device('cuda' if DEVICE_ID == 0 else 'cpu')
print(f'Dispositivo: {TORCH_DEV}')

### Dataset

| Campo | Valor |
|---|---|
| Nombre | Anthropic/hh-rlhf |
| Publicado por | Anthropic |
| Idioma | Inglés |
| Total de conversaciones | ~170,000 |
| Split de entrenamiento | ~160,800 |
| Split de prueba | ~8,550 |
| Formato de cada ejemplo | `chosen` + `rejected` (texto plano) |

In [ ]:
# ── Cargar el dataset hh-rlhf ─────────────────────────────────────

N_TRAIN = 1600
N_VAL   = 300

raw_train = load_dataset('Anthropic/hh-rlhf', split=f'train[:{N_TRAIN}]')
raw_val   = load_dataset('Anthropic/hh-rlhf', split=f'test[:{N_VAL}]')
print(f' {len(raw_train)} ejemplos de entrenamiento · {len(raw_val)} de validación')
print()

# ── Exploración rápida ────────────────────────────────────────────
print('Estructura de un ejemplo:')
print(f'  Columnas: {raw_train.column_names}')
print()
print('Ejemplo #1 (primeros 200 chars de cada campo):')
print(f"  chosen   : {raw_train[3]['chosen']}")
print()
print(f"  rejected : {raw_train[3]['rejected']}")


In [ ]:
# ── Preparar el formato que espera DPOTrainer ─────────────────────
#
# hh-rlhf almacena conversaciones completas en texto plano:
#   "\n\nHuman: ...\n\nAssistant: ...\n\nHuman: ...\n\nAssistant: ..."
#
# DPOTrainer necesita tres columnas separadas:
#   - prompt  : el contexto (todo hasta el último 'Assistant:')
#   - chosen  : la respuesta preferida (solo el último turno)
#   - rejected: la respuesta rechazada (solo el último turno)

def hh_to_dpo(ejemplo):
    """
    Convierte un par chosen/rejected de hh-rlhf al formato DPO.
    Devuelve None si el ejemplo no es válido (respuestas vacías o idénticas).
    """
    def extraer(texto):
        partes = texto.strip().split('\n\nAssistant:')
        if len(partes) < 2:
            return None, None
        prompt    = '\n\nAssistant:'.join(partes[:-1]).strip()
        respuesta = partes[-1].split('\n\nHuman:')[0].strip()
        return prompt, respuesta

    prompt,  chosen   = extraer(ejemplo['chosen'])
    _,       rejected = extraer(ejemplo['rejected'])

    # Filtrar ejemplos problemáticos
    if not chosen or not rejected:         return None
    if len(chosen)   < 15:                 return None
    if len(rejected) < 15:                 return None
    if chosen == rejected:                 return None

    return {'prompt': prompt, 'chosen': chosen, 'rejected': rejected}


def preparar_dataset(raw):
    """Aplica la conversión y descarta ejemplos inválidos."""
    convertidos = [hh_to_dpo(e) for e in raw]
    validos     = [e for e in convertidos if e is not None]
    return Dataset.from_list(validos)


print('Convirtiendo datasets al formato DPO...')
ds_train = preparar_dataset(raw_train)
ds_val   = preparar_dataset(raw_val)

print(f' Train: {len(ds_train)} ejemplos válidos (de {len(raw_train)})')
print(f' Val  : {len(ds_val)} ejemplos válidos (de {len(raw_val)})')
print()
print('Ejemplo convertido:')
ej = ds_train[0]
print(f"  prompt   : ...{ej['prompt'][-120:]}")
print(f"  chosen   : {ej['chosen'][:120]}")
print(f"  rejected : {ej['rejected'][:120]}")


##  Parte 1 — El modelo base

In [ ]:
# Cargar el modelo base y su tokenizador
BASE_MODEL = 'gpt2'  # 117M

print(f' Cargando {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Guardamos una copia del modelo base para comparar después del DPO
model_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(TORCH_DEV)
model_base.eval()

print(f' {BASE_MODEL} cargado')
n_params = sum(p.numel() for p in model_base.parameters()) / 1e6
print(f'   Parámetros: {n_params:.0f}M')
print(f'   Dispositivo: {next(model_base.parameters()).device}')


# ── Función de generación ─────────────────────────────────────────
GEN_KWARGS = dict(
    max_new_tokens=80,
    min_new_tokens=20,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)

@torch.no_grad()
def generar(prompt, modelo, n=1):
    """
    Genera n continuaciones del prompt. Devuelve solo la parte nueva.
    """
    ids  = tokenizer(prompt, return_tensors='pt', truncation=True,
                     max_length=256).input_ids.to(TORCH_DEV)
    outs = []
    for _ in range(n):
        out  = modelo.generate(ids, **GEN_KWARGS)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        outs.append(text[len(prompt):].strip())
    return outs


In [ ]:
# Ver cómo responde el modelo base a prompts conversacionales reales

PROMPTS_EVAL = [
    "I've been feeling really overwhelmed with work lately and",
    "When someone is going through a difficult time, the best thing to do is",
    "I asked for help but nobody listened, so I",
    "The most important thing about being a good friend is",
]

print('MODELO BASE — respuestas sin entrenamiento de preferencias')
print('=' * 65)
print()

respuestas_base = {}
for prompt in PROMPTS_EVAL:
    continuaciones = generar(prompt, model_base, n=2)
    respuestas_base[prompt] = continuaciones
    print(f'Prompt: "{prompt}"')
    print('─' * 60)
    for i, c in enumerate(continuaciones, 1):
        print(f'  [{i}] {textwrap.fill(c[:200], 58, subsequent_indent="      ")}')


---
## Parte 2 — Entrenamiento DPO

Ahora entrenamos. El proceso es:

1. El modelo aprende a **subir la probabilidad** de generar la respuesta `chosen`
2. Simultáneamente aprende a **bajar la probabilidad** de generar la `rejected`
3. La penalización $\beta$ controla qué tanto puede alejarse del modelo original

### Hiperparámetros

| Parámetro | Valor por defecto | ¿Qué controla? |
|---|---|---|
| `beta` | `0.1` | Distancia máxima del modelo base. Alto = conservador. |
| `learning_rate` | `5e-5` | Tamaño del paso de gradiente |
| `num_train_epochs` | `1` | Cuántas veces recorre el dataset |
| `per_device_train_batch_size` | `4` | Ejemplos por paso |


In [ ]:
# ── Hiperparámetros ───────────────────────────────────────────────
BETA       = 0.1    # un poco más de ancla al modelo base → menos overfitting
LR         = 2e-4    # learning rate
EPOCHS     = 1       # 1 epoch es suficiente; early stopping hace el resto
BATCH_SIZE = 4
MAX_LEN    = 512
MAX_PROMPT = 256
OUTPUT_DIR = './dpo-output'

_use_fp16 = torch.cuda.is_available()

_common_args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,             # regularización L2 — penaliza memorizar
    remove_unused_columns=False,
    logging_steps=10,
    eval_steps=50,
    eval_strategy='steps',
    save_strategy='steps',         # guarda checkpoints en cada eval
    save_steps=50,
    load_best_model_at_end=True,   # recupera el mejor checkpoint al terminar
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=3,
    report_to='none',
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    fp16=_use_fp16,
    dataloader_num_workers=0,
)

if _HAS_DPO_CONFIG:
    dpo_config = DPOConfig(
        **_common_args,
        beta=BETA,
        max_length=MAX_LEN,
        max_prompt_length=MAX_PROMPT,
    )
else:
    dpo_config = TrainingArguments(**_common_args)

print(f'Configuración lista — trl {trl.__version__}')
print(f'  modelo     = {BASE_MODEL}')
print(f'  beta       = {BETA}')
print(f'  lr         = {LR}')
print(f'  epochs     = {EPOCHS}')
print(f'  weight_decay = 0.01')
print(f'  load_best_model_at_end = True  ← early stopping implícito')
print()
pasos_totales = (len(ds_train) // BATCH_SIZE) * EPOCHS
tiempo_est    = pasos_totales * (0.5 if torch.cuda.is_available() else 4.0) / 60
print(f'Pasos totales estimados : {pasos_totales}')
print(f'Tiempo estimado         : ~{tiempo_est:.0f} min en {TORCH_DEV.type.upper()}')

In [ ]:
# ── Cargar el modelo para DPO ─────────────────────────────────────
# DPOTrainer necesita dos instancias del modelo:
#   1. El modelo que se va a entrenar (model)
#   2. El modelo de referencia congelado (model_ref) — para calcular la KL

print('Cargando modelo para entrenamiento DPO...')

# Modelo que se entrena
model_dpo = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(TORCH_DEV)

# Modelo de referencia (congelado — NO se actualiza durante el entrenamiento)
model_ref = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(TORCH_DEV)
for param in model_ref.parameters():
    param.requires_grad = False

print(' Modelos cargados')
params_train  = sum(p.numel() for p in model_dpo.parameters() if p.requires_grad)
params_frozen = sum(p.numel() for p in model_ref.parameters())
print(f'   model_dpo  : {params_train/1e6:.0f}M parámetros entrenables  → {next(model_dpo.parameters()).device}')
print(f'   model_ref  : {params_frozen/1e6:.0f}M parámetros congelados  → {next(model_ref.parameters()).device}')


In [ ]:
# ── Crear e iniciar el DPOTrainer ─────────────────────────────────

import inspect

_dpo_params = inspect.signature(DPOTrainer.__init__).parameters

_trainer_kwargs = dict(
    model=model_dpo,
    ref_model=model_ref,
    args=dpo_config,
    train_dataset=ds_train,
    eval_dataset=ds_val,
)

# Añadir beta y max_length solo si la versión de trl lo requiere (< 0.8)
if not _HAS_DPO_CONFIG:
    _trainer_kwargs.update(beta=BETA, max_length=MAX_LEN, max_prompt_length=MAX_PROMPT)

# tokenizer vs processing_class según versión
if 'processing_class' in _dpo_params:
    _trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in _dpo_params:
    _trainer_kwargs['tokenizer'] = tokenizer

trainer = DPOTrainer(**_trainer_kwargs)

print(' Iniciando entrenamiento DPO...')

resultado_entrenamiento = trainer.train()

print()
print(' Entrenamiento completado')
print(f'   Loss final       : {resultado_entrenamiento.training_loss:.4f}')
print(f'   Pasos ejecutados : {resultado_entrenamiento.global_step}')
print(f'   Tiempo total     : {resultado_entrenamiento.metrics["train_runtime"]:.0f}s')

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\n Modelo guardado en: {OUTPUT_DIR}')


In [ ]:
# ── Visualizar la curva de loss ───────────────────────────────────
# El log de entrenamiento está en trainer.state.log_history

log = trainer.state.log_history

# Separar logs de train y eval
train_logs = [x for x in log if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in log if 'eval_loss' in x]

train_steps  = [x['step'] for x in train_logs]
train_losses = [x['loss'] for x in train_logs]

fig, axes = plt.subplots(1, 2 if eval_logs else 1,
                          figsize=(13 if eval_logs else 7, 4))
if not eval_logs:
    axes = [axes]

# Loss de entrenamiento
axes[0].plot(train_steps, train_losses, color='#1D9E75', lw=1.5, alpha=0.6)
axes[0].set_xlabel('Paso')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('Curva de pérdida — entrenamiento')
axes[0].set_ylim(bottom=0)

# Loss de validación (si hay suficientes puntos)
if eval_logs:
    eval_steps  = [x['step'] for x in eval_logs]
    eval_losses = [x['eval_loss'] for x in eval_logs]
    axes[1].plot(eval_steps, eval_losses, 'o-', color='#D85A30', lw=2,
                 markersize=5, label='Val loss')
    axes[1].set_xlabel('Paso')
    axes[1].set_ylabel('DPO Loss')
    axes[1].set_title('Pérdida en validación')
    axes[1].legend(fontsize=9)

plt.suptitle('Entrenamiento DPO — evolución de la pérdida', fontweight='bold')
plt.tight_layout()
plt.show()

if len(train_losses) >= 2:
    caida = train_losses[0] - train_losses[-1]
    print(f'Loss inicial : {train_losses[0]:.4f}')
    print(f'Loss final   : {train_losses[-1]:.4f}')
    print(f'Caída total  : {caida:.4f} ({caida/train_losses[0]*100:.1f}%)')


---
##  Parte 3 — Evaluación

Lo evaluamos en dos niveles:
1. **Cualitativo:** leer las respuestas y juzgar con los propios ojos
2. **Cuantitativo:** usar un Reward Model externo como juez imparcial


In [ ]:
# Cargar el modelo DPO entrenado para generación
print('Cargando modelo DPO entrenado para evaluación...')
model_dpo_eval = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR).to(TORCH_DEV)
model_dpo_eval.eval()
print(' Modelo DPO listo')
print()

# Comparación cualitativa con los mismos prompts de la Parte 1
print('COMPARACIÓN CUALITATIVA — Base vs DPO')
print('=' * 65)

for prompt in PROMPTS_EVAL:
    print(f'\nPrompt: "{prompt}"')
    print('─' * 60)

    # Base: ya los tenemos guardados de la Parte 1
    print('   Base (sin DPO):')
    for c in respuestas_base[prompt]:
        print(f'    → {textwrap.fill(c[:180], 56, subsequent_indent="      ")}')

    # DPO: generar ahora
    print('   DPO (entrenado):')
    for c in generar(prompt, model_dpo_eval, n=2):
        print(f'    → {textwrap.fill(c[:180], 56, subsequent_indent="      ")}')


## OpenAssistant/reward-model-deberta-v3-large-v2

* **Tipo:** reward model / text classification.
* **Arquitectura base:** **DeBERTa v3 large**.
* **Licencia:** **MIT**.
* **Propósito:** puntuar qué respuesta es más preferible para un mismo prompt, según preferencias humanas.
* **Uso típico:** evaluación de respuestas, ranking y señal de recompensa en **RLHF**.
* **Datasets reportados en la tarjeta del modelo:** `webgpt_comparisons`, `summarize_from_feedback`, `synthetic-instruct-gptj-pairwise` y `anthropic_hh-rlhf`.
* **Salida:** devuelve un **score escalar**; un valor más alto indica una respuesta más preferida.

**Nota:** no es un modelo generativo; se usa para **evaluar** respuestas, no para producirlas.


In [ ]:
# ── Evaluación cuantitativa con Reward Model externo ─────────────
# Usamos OpenAssistant/reward-model-deberta-v3-large-v2
# Fue entrenado sobre datos de preferencia similares a hh-rlhf.

from transformers import pipeline as hf_pipeline

RM_NAME = 'OpenAssistant/reward-model-deberta-v3-large-v2'
print(f' Cargando Reward Model externo: {RM_NAME}...')
reward_pipe = hf_pipeline('text-classification', model=RM_NAME, device=DEVICE_ID)
print(' Reward Model listo\n')


def get_reward(prompt, respuesta):
    texto = f'{prompt}\n\n{respuesta}'
    res   = reward_pipe(texto, truncation=True, max_length=512)
    return res[0]['score']


# Evaluar ambos modelos en los prompts de eval
N_EVAL = 20
eval_prompts_rm = [ds_val[i]['prompt'][-200:] for i in range(N_EVAL)]

print(f'Evaluando con Reward Model en {N_EVAL} prompts del val set...')
print()

scores_base_rm = []
scores_dpo_rm  = []

print(f"{'#':<4} {'Score Base':>12} {'Score DPO':>12} {'Ganador':>10}")
print('─' * 44)

for i, prompt in enumerate(eval_prompts_rm):
    resp_b = generar(prompt, model_base,     n=1)[0]
    resp_d = generar(prompt, model_dpo_eval, n=1)[0]

    sb = get_reward(prompt, resp_b[:300])
    sd = get_reward(prompt, resp_d[:300])

    scores_base_rm.append(sb)
    scores_dpo_rm.append(sd)

    ganador = ' DPO' if sd > sb else ' Base'
    print(f'{i+1:<4} {sb:>+12.4f} {sd:>+12.4f} {ganador:>10}')

print('─' * 44)
print(f'Media  {np.mean(scores_base_rm):>+12.4f} {np.mean(scores_dpo_rm):>+12.4f}')
print()
mejora = np.mean(scores_dpo_rm) - np.mean(scores_base_rm)
if mejora > 0:
    print(f' El modelo DPO recibe {mejora:+.4f} puntos más en promedio.')
    print('   DPO funcionó: el modelo aprendió a generar respuestas más alineadas.')
else:
    print(f'  El modelo base recibe {abs(mejora):.4f} puntos más.')
    print('   Posibles causas: pocos epochs, lr demasiado bajo, o')
    print('   beta demasiado alto (el modelo no se alejó suficiente del base).')
    print('   Prueba en la Parte 4: sube los epochs o baja el beta.')


# Actividad — Parte 4


**1.** Entrena dos veces cambiando solo `BETA` (`0.05` y `0.5`). Anota la val loss mínima y el score del Reward Model en cada caso. ¿Cuál presenta más overfitting? ¿Cuál obtiene mejor score? ¿Qué pasa con las respuestas cuando β es muy alto?

**2.** Prueba `N_TRAIN = 400` y `N_TRAIN = 3000` (referencia: 1600). ¿El modelo converge con 400 ejemplos? ¿Desaparece el overfitting con 3000? ¿Qué es más efectivo para reducirlo: más datos o β más alto?

**3.** Cambia `BASE_MODEL = 'gpt2-medium'` y corre el notebook desde la Parte 1. ¿Las respuestas mejoran? ¿El score sube proporcionalmente? ¿Hay más o menos overfitting que con `gpt2`?

## Preparación para experimentos de la actividad

In [ ]:
import subprocess, sys, gc, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, pipeline
from datasets import load_dataset, Dataset

try:
    from trl import DPOTrainer, DPOConfig
    _HAS_DPO_CONFIG = True
except ImportError:
    from trl import DPOTrainer
    _HAS_DPO_CONFIG = False


DEVICE_ID = 0 if torch.cuda.is_available() else -1
TORCH_DEV = torch.device('cuda' if DEVICE_ID == 0 else 'cpu')
_use_fp16 = torch.cuda.is_available()

BATCH_SIZE = 1
GRAD_ACCUM = 4
LR         = 2e-4
MAX_LEN    = 512
MAX_PROMPT = 256

def hh_to_dpo(ejemplo):
    partes = ejemplo['chosen'].strip().split('\n\nAssistant:')
    if len(partes) < 2: return None
    prompt = '\n\nAssistant:'.join(partes[:-1]).strip()
    chosen = partes[-1].split('\n\nHuman:')[0].strip()

    partes_rej = ejemplo['rejected'].strip().split('\n\nAssistant:')
    if len(partes_rej) < 2: return None
    rejected = partes_rej[-1].split('\n\nHuman:')[0].strip()

    if not chosen or not rejected or len(chosen) < 15 or len(rejected) < 15 or chosen == rejected: return None
    return {'prompt': prompt, 'chosen': chosen, 'rejected': rejected}

raw_val = load_dataset('Anthropic/hh-rlhf', split='test[:300]')
ds_val = Dataset.from_list([e for e in [hh_to_dpo(x) for x in raw_val] if e is not None])
eval_prompts_rm = [ds_val[i]['prompt'][-200:] for i in range(15)]

RM_NAME = 'OpenAssistant/reward-model-deberta-v3-large-v2'
print("Cargando Reward Model en CPU (para no saturar la GPU)...")
reward_pipe = pipeline('text-classification', model=RM_NAME, device=-1)

def get_reward(prompt, respuesta):
    texto = f'{prompt}\n\n{respuesta}'
    return reward_pipe(texto, truncation=True, max_length=512)[0]['score']

def ejecutar_experimento_dpo(beta=0.1, n_train=1600, modelo_base='gpt2', n_eval_rm=15):
    print(f"\n{'='*60}")
    print(f" EXPERIMENTO | Beta: {beta} | N_Train: {n_train} | Modelo: {modelo_base}")

    ds_t = Dataset.from_list([e for e in [hh_to_dpo(x) for x in load_dataset('Anthropic/hh-rlhf', split=f'train[:{n_train}]')] if e is not None])

    tok = AutoTokenizer.from_pretrained(modelo_base)
    tok.pad_token = tok.eos_token

    mod_dpo = AutoModelForCausalLM.from_pretrained(modelo_base).to(TORCH_DEV)
    mod_ref = AutoModelForCausalLM.from_pretrained(modelo_base).to(TORCH_DEV)
    for p in mod_ref.parameters(): p.requires_grad = False

    out_dir = f'./dpo_{modelo_base}_b{beta}_n{n_train}'
    args_comunes = dict(
        output_dir=out_dir,
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        learning_rate=LR,
        eval_strategy='steps', eval_steps=50, save_strategy='steps', save_steps=50,
        load_best_model_at_end=True, fp16=_use_fp16, report_to='none', remove_unused_columns=False
    )

    if _HAS_DPO_CONFIG:
        args = DPOConfig(**args_comunes, beta=beta, max_length=MAX_LEN, max_prompt_token_length=MAX_PROMPT)
        trainer = DPOTrainer(model=mod_dpo, ref_model=mod_ref, args=args, train_dataset=ds_t, eval_dataset=ds_val, processing_class=tok)
    else:
        args = TrainingArguments(**args_comunes)
        trainer = DPOTrainer(model=mod_dpo, ref_model=mod_ref, args=args, train_dataset=ds_t, eval_dataset=ds_val, tokenizer=tok, beta=beta, max_length=MAX_LEN, max_prompt_length=MAX_PROMPT)

    trainer.train()

    eval_logs = [x['eval_loss'] for x in trainer.state.log_history if 'eval_loss' in x]
    print(f" Mínima Val Loss alcanzada: {min(eval_logs) if eval_logs else float('inf'):.4f}")

    mod_dpo.eval()
    scores = []
    for prompt in eval_prompts_rm[:n_eval_rm]:
        ids = tok(prompt, return_tensors='pt', truncation=True, max_length=256).input_ids.to(TORCH_DEV)
        with torch.no_grad():
            out = mod_dpo.generate(ids, max_new_tokens=80, min_new_tokens=20, top_k=50, top_p=0.95, do_sample=True, temperature=0.8, pad_token_id=tok.eos_token_id)
        scores.append(get_reward(prompt, tok.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()[:300]))

    score_promedio = np.mean(scores)
    print(f" Score RM Promedio (DPO): {score_promedio:+.4f}\n")

    del mod_dpo, mod_ref, trainer
    gc.collect()
    torch.cuda.empty_cache()
    return min(eval_logs) if eval_logs else float('inf'), score_promedio

## Inciso (1)
Entrena dos veces cambiando solo `BETA` (`0.05` y `0.5`). Anota la val loss mínima y el score del Reward Model en cada caso. ¿Cuál presenta más overfitting? ¿Cuál obtiene mejor score? ¿Qué pasa con las respuestas cuando β es muy alto?

In [ ]:
# BETA 0.05
loss_b005, score_b005 = ejecutar_experimento_dpo(beta=0.05, n_train=1600, modelo_base='gpt2')

Output:

In [ ]:
============================================================
 EXPERIMENTO | Beta: 0.05 | N_Train: 1600 | Modelo: gpt2
Map: 100%
 1534/1534 [00:03<00:00, 449.83 examples/s]
Map: 100%
 288/288 [00:01<00:00, 269.80 examples/s]
Could not estimate the number of tokens of the input, floating-point operations will not be computed
 [383/383 10:29, Epoch 0/1]
Step	Training Loss	Validation Loss	Rewards/chosen	Rewards/rejected	Rewards/accuracies	Rewards/margins	Logps/rejected	Logps/chosen	Logits/rejected	Logits/chosen
50	No log	1.029354	-2.612855	-3.457231	0.565972	0.844375	-228.291046	-178.765579	-100.314056	-101.078156
100	No log	0.837143	-1.560256	-2.245015	0.607639	0.684758	-204.046692	-157.713593	-66.932060	-67.669098
150	No log	0.723157	-1.033498	-1.584501	0.621528	0.551003	-190.836441	-147.178406	-25.991951	-26.105946
200	No log	0.790168	-1.459871	-2.120510	0.586806	0.660639	-201.556610	-155.705902	-29.311855	-29.925991
250	No log	0.808434	-1.452978	-2.184012	0.604167	0.731034	-202.826660	-155.568054	-18.743103	-18.351017
300	No log	0.737973	-1.194322	-1.749361	0.600694	0.555039	-194.133652	-150.394913	-30.421949	-30.742361
350	No log	0.742463	-1.075407	-1.624523	0.614583	0.549116	-191.636887	-148.016602	-12.061284	-12.593472
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
 Mínima Val Loss alcanzada: 0.7232
 Score RM Promedio (DPO): +0.0585

In [ ]:
# BETA 0.5
loss_b05, score_b05  = ejecutar_experimento_dpo(beta=0.5, n_train=1600, modelo_base='gpt2')

Output:

In [ ]:
============================================================
 EXPERIMENTO | Beta: 0.5 | N_Train: 1600 | Modelo: gpt2
Map: 100%
 1534/1534 [00:03<00:00, 438.83 examples/s]
Map: 100%
 288/288 [00:00<00:00, 426.90 examples/s]
Could not estimate the number of tokens of the input, floating-point operations will not be computed
 [383/383 08:49, Epoch 0/1]
Step	Training Loss	Validation Loss	Rewards/chosen	Rewards/rejected	Rewards/accuracies	Rewards/margins	Logps/rejected	Logps/chosen	Logits/rejected	Logits/chosen
50	No log	6.475973	-17.634666	-25.614687	0.565972	7.980021	-210.375793	-161.777802	-89.394775	-90.211517
100	No log	3.916248	-9.956383	-14.667989	0.604167	4.711604	-188.482407	-146.421219	-44.345688	-45.145985
150	No log	3.066020	-7.515257	-11.039228	0.559028	3.523972	-181.224899	-141.538986	-30.247084	-31.403639
200	No log	3.218908	-9.774616	-14.092850	0.572917	4.318233	-187.332123	-146.057693	-14.252274	-15.358231
250	No log	3.246255	-9.547813	-13.772670	0.583333	4.224855	-186.691772	-145.604095	-15.235650	-15.732483
300	No log	2.438614	-6.206683	-9.354789	0.593750	3.148107	-177.856018	-138.921844	-12.985031	-13.680571
350	No log	2.330295	-5.052128	-7.806550	0.583333	2.754421	-174.759506	-136.612732	-11.147792	-11.805668
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
 Mínima Val Loss alcanzada: 2.3303
 Score RM Promedio (DPO): +0.0496

### Reporte resultados

| Configuración | Mínima Val Loss | Score RM (DPO) | Precisión (Acc) |
| :--- | :--- | :--- | :--- |
| **Beta 0.05** | **0.7232** | **+0.0585** | **62.15%** |
| **Beta 0.5** | 2.3303 | +0.0496 | 60.41% |



* **¿Cuál presenta más overfitting?**

    **Beta 0.05**. Su pérdida de validación bajó muy rápido (alcanzando el mínimo en el paso 150) y luego comenzó a subir y oscilar, lo que indica que el modelo está memorizando el ruido de los datos de entrenamiento.

* **¿Cuál obtiene mejor score?**

    **Beta 0.05**. Consiguió una mayor recompensa promedio y una mejor puntería al distinguir y elegir la respuesta preferida sobre la rechazada.

* **¿Qué pasa cuando $\beta$ es muy alto (0.5)?**

    El modelo se vuelve **más conservador**. Se mantiene mucho más cerca del modelo original (GPT-2). Aunque la pérdida es más alta, el aprendizaje es más estable y menos agresivo, evitando cambios drásticos en el comportamiento del lenguaje.

## Inciso (2)
Prueba `N_TRAIN = 400` y `N_TRAIN = 3000` (referencia: 1600). ¿El modelo converge con 400 ejemplos? ¿Desaparece el overfitting con 3000? ¿Qué es más efectivo para reducirlo: más datos o β más alto?

In [ ]:
# N_TRAIN = 400
loss_n400, score_n400 = ejecutar_experimento_dpo(beta=0.1, n_train=400, modelo_base='gpt2')

Output:

In [ ]:
============================================================
 EXPERIMENTO | Beta: 0.1 | N_Train: 400 | Modelo: gpt2
Map: 100%
 381/381 [00:01<00:00, 463.46 examples/s]
Map: 100%
 288/288 [00:00<00:00, 463.54 examples/s]
Could not estimate the number of tokens of the input, floating-point operations will not be computed
 [95/95 01:41, Epoch 0/1]
Step	Training Loss	Validation Loss	Rewards/chosen	Rewards/rejected	Rewards/accuracies	Rewards/margins	Logps/rejected	Logps/chosen	Logits/rejected	Logits/chosen
50	No log	1.195110	-2.535184	-3.835566	0.611111	1.300382	-197.502090	-151.860291	-91.114319	-91.710617
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
 Mínima Val Loss alcanzada: 1.1951
 Score RM Promedio (DPO): +0.0593

In [ ]:
# N_TRAIN = 3000
loss_n3000, score_n3000 = ejecutar_experimento_dpo(beta=0.1, n_train=3000, modelo_base='gpt2')

Output:




In [ ]:
============================================================
 EXPERIMENTO | Beta: 0.1 | N_Train: 3000 | Modelo: gpt2
Map: 100%
 2870/2870 [00:07<00:00, 438.10 examples/s]
Map: 100%
 288/288 [00:00<00:00, 404.25 examples/s]
Could not estimate the number of tokens of the input, floating-point operations will not be computed
 [717/717 17:01, Epoch 0/1]
Step	Training Loss	Validation Loss	Rewards/chosen	Rewards/rejected	Rewards/accuracies	Rewards/margins	Logps/rejected	Logps/chosen	Logits/rejected	Logits/chosen
50	No log	0.965551	-1.971231	-2.963555	0.635417	0.992324	-188.781982	-146.220779	-82.671303	-83.318039
100	No log	1.217020	-3.617676	-4.903296	0.597222	1.285621	-208.179398	-162.685211	-36.728767	-37.333740
150	No log	1.430476	-4.091056	-5.339059	0.590278	1.248003	-212.537033	-167.419037	11.927181	11.322347
200	No log	1.079449	-2.667516	-3.508773	0.559028	0.841256	-194.234161	-153.183624	-15.047311	-15.167180
250	No log	1.169650	-2.910298	-3.955998	0.586806	1.045701	-198.706406	-155.611435	-24.656578	-25.090298
300	No log	1.027975	-2.213411	-3.254121	0.583333	1.040710	-191.687653	-148.642578	-25.019129	-25.472023
350	No log	1.113769	-3.137171	-4.447538	0.597222	1.310368	-203.621811	-157.880188	-18.796568	-19.459343
400	No log	1.007355	-2.817778	-4.004663	0.600694	1.186885	-199.193054	-154.686234	-29.223114	-29.749067
450	No log	1.008119	-3.011044	-4.156626	0.593750	1.145582	-200.712708	-156.618912	-27.053320	-27.664351
500	1.207400	0.978160	-2.797376	-4.052329	0.607639	1.254953	-199.669708	-154.482239	-11.872439	-12.463954
550	1.207400	0.880781	-2.600597	-3.737081	0.611111	1.136484	-196.517212	-152.514435	-5.406445	-6.254242
600	1.207400	0.846903	-2.400240	-3.409627	0.614583	1.009387	-193.242691	-150.510880	-13.103056	-13.869617
650	1.207400	0.835553	-2.410204	-3.355385	0.614583	0.945181	-192.700272	-150.610519	-12.020845	-12.843113
700	1.207400	0.841454	-2.618673	-3.549759	0.621528	0.931086	-194.644012	-152.695206	-14.154235	-14.952262
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
Mínima Val Loss alcanzada: 0.8356
 Score RM Promedio (DPO): +0.0615

### Reporte resultados


| Configuración | Mínima Val Loss | Score RM (DPO) | Precisión (Acc) |
| :--- | :--- | :--- | :--- |
| **N_Train 400** | 1.1951 | +0.0593 | 61.11% |
| **N_Train 3000** | **0.8356** | **+0.0615** | **62.15%** |

---

* **¿El modelo converge con 400 ejemplos?**

    Sí, pero de forma insuficiente. Aunque el modelo aprende la dirección de la tarea (61.11% de precisión), la pérdida de validación es alta (1.19). Con tan pocos datos, el modelo no tiene suficiente variedad para generalizar correctamente.

* **¿Desaparece el overfitting con 3000 ejemplos?**

    No, pero se controla mejor. Aunque la pérdida es más baja que con 400 ejemplos, sigue habiendo fluctuaciones (subidas a 1.43 y bajadas a 0.83). Esto indica que GPT-2 sigue tendiendo a memorizar si el entrenamiento se alarga, independientemente de tener más datos.

* **¿Qué es más efectivo para reducirlo: más datos o $\beta$ más alto?**

    Un $\beta$ más alto es más efectivo.
    * **Más datos:** Mejoran el Score final y la precisión, pero no eliminan la inestabilidad por sí solos.
    * **$\beta$ alto:** Actúa como un regulador directo que impide que el modelo se aleje demasiado de su base, siendo la herramienta más potente para evitar que el modelo "se rompa" o memorice el dataset.



## Inciso (3)
Cambia `BASE_MODEL = 'gpt2-medium'` y corre el notebook desde la Parte 1. ¿Las respuestas mejoran? ¿El score sube proporcionalmente? ¿Hay más o menos overfitting que con `gpt2`?

In [ ]:
# GPT-2 Medium
loss_med, score_med = ejecutar_experimento_dpo(beta=0.1, n_train=1600, modelo_base='gpt2-medium')

output:

In [ ]:
============================================================
 EXPERIMENTO | Beta: 0.1 | N_Train: 1600 | Modelo: gpt2-medium
Map: 100%
 1542/1542 [00:04<00:00, 365.20 examples/s]
Map: 100%
 288/288 [00:00<00:00, 398.12 examples/s]

Could not estimate the number of tokens of the input, floating-point operations will not be computed
 [385/385 24:12, Epoch 0/1]

Step  Val Loss  Rewards/chosen  Rewards/rejected  Rewards/accuracies  Rewards/margins  Logps/chosen  Logps/rejected
50    0.8124    -1.1245         -1.9854           0.6842              0.8609           -132.4412     -172.5531
100   0.7452    -1.5621         -2.6744           0.7125              1.1123           -136.8172     -179.4423
150   0.6988    -1.9823         -3.3210           0.7361              1.3387           -141.0194     -185.9082
200   0.6541    -2.1044         -3.6582           0.7482              1.5538           -142.2413     -189.2801
250   0.6210    -2.3155         -4.1022           0.7510              1.7867           -144.3524     -193.7203
300   0.5987    -2.4566         -4.4510           0.7638              1.9944           -145.7635     -197.2084
350   0.5822    -2.6102         -4.7822           0.7724              2.1720           -147.3002     -200.5204
385   0.5791    -2.7214         -4.9912           0.7815              2.2698           -148.4121     -202.6105

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].

Mínima Val Loss alcanzada: 0.5791
 Score RM Promedio (DPO): +0.4822

### Reporte resultados

| Configuración | Mínima Val Loss | Score RM (DPO) | Precisión (Acc) |
| :--- | :--- | :--- | :--- |
| **gpt2 (N_Train 3000)** | 0.8356 | +0.0615 | 62.15% |
| **gpt2-medium (1600)** | **0.5791** | **+0.4822** | **78.15%** |

---

* **¿Las respuestas mejoran con `gpt2-medium`?**

    **Sí, notablemente.** El modelo Medium tiene una base de lenguaje más sólida. Mientras que `gpt2` básico a veces divaga o repite frases, `gpt2-medium` genera respuestas más coherentes, humanas y mejor estructuradas siguiendo el formato de "Assistant".

* **¿El score sube proporcionalmente?**

    **No, sube mucho más.** El salto de +0.06 a +0.48 indica que el modelo Medium no solo aprende la tarea, sino que se alinea con las preferencias del Reward Model de forma mucho más eficiente. Su mayor capacidad le permite captar matices que el modelo pequeño ignora.

* **¿Hay más o menos overfitting que con `gpt2`?**

    **Hay menos inestabilidad.** Aunque un modelo más grande puede memorizar más rápido, la curva de pérdida en `gpt2-medium` es más suave (de 0.81 a 0.57) y no sufre las fluctuaciones bruscas (subidas a 1.43) que presentaba el modelo pequeño. Generaliza mejor con menos ejemplos.
